This script imports essential libraries for geospatial data analysis, visualization, and clustering. GeoPandas and Pandas handle spatial and tabular data, while Shapely supports geometry creation and operations. Matplotlib provides static plotting, and Plotly enables interactive charts and dashboards. Folium creates web-based interactive maps with tooltips and temporal animations via TimestampedGeoJson. NumPy facilitates numerical operations. Together, these tools allow loading, processing, analyzing, and visualizing geospatial datasets, enabling spatial clustering, interactive mapping, and advanced visual storytelling.

In [ ]:
 #For Data ---
import pandas as pd
import numpy as np

# --- Geo ---
import geopandas as gpd
from shapely import speedups as _shp_speedups
from shapely.geometry import box, Polygon

# --- Viz (static + interactive) ---
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# --- Maps ---
import folium
from folium.features import GeoJsonTooltip
from folium.plugins import TimestampedGeoJson


In [ ]:
!pip install -q "folium>=0.12" mapclassify matplotlib

# **DATA UNDERSTANDING**

**a) Deforestation Data**

This line reads the DETER shapefile containing Amazon deforestation alerts into a GeoDataFrame using GeoPandas. The gpd.read_file() function loads both attribute data (e.g., date, alert type) and geometric information (polygons) from the file path provided (/content/drive/My Drive/Asim/Dissertation/deter-amz-deter-public.shp) into the variable Deter, enabling spatial analysis, visualization, and further processing in Python.

In [ ]:
Deter=gpd.read_file('/content/drive/My Drive/Asim/Dissertation/deter-amz-deter-public.shp')

This dataset, loaded into the Deter GeoDataFrame, contains deforestation and degradation alerts from the Amazon DETER system. Each row represents a detected area with attributes such as FID (feature ID), CLASSNAME (alert type like DESMATAMENTO_CR or CICATRIZ_DE_QUEIMADA), VIEW_DATE (observation date), sensor and satellite used (e.g., WFI, AMAZONIA-1), and location details including municipality name, IBGE code, and state (UF). The AREAMUNKM field indicates affected area in square kilometers, and geometry stores the polygon shape representing the alert location. This structure allows spatial analysis, clustering, and visualization of deforestation events by time, location, or category.

In [ ]:
Deter.head()

These values represent the frequency of different alert classes within the DETER dataset. The majority of alerts are DESMATAMENTO_CR (clear-cut deforestation) with 251,895 occurrences, followed by CICATRIZ_DE_QUEIMADA (burn scars) with 114,482. Other significant categories include DEGRADACAO (forest degradation, 36,597), DESMATAMENTO_VEG (vegetation removal, 9,334), and MINERACAO (mining, 7,679). Smaller but notable classes are CS_DESORDENADO (disordered selective cutting, 5,690), CS_GEOMETRICO (geometric selective cutting, 4,124), and CORTE_SELETIVO (selective logging). This distribution highlights that clear-cut deforestation and burn scars dominate Amazon forest disturbances, while mining and selective logging occur at a smaller scale.

In [ ]:
Deter['CLASSNAME'].value_counts()

The dataset includes observations captured by multiple Earth observation satellites used in the DETER system. The primary satellites are AMAZONIA-1, Brazil’s first satellite dedicated to environmental monitoring, and CBERS-4/CBERS-4A, which are part of the China-Brazil Earth Resources Satellite program. Additionally, RESOURCESAT-2, an Indian remote sensing satellite, contributes to the imagery. Some variations in naming, such as Amazonia-1 and CBER-4A, are present due to inconsistencies in data entry and should be standardized for accurate analysis. These satellites provide multispectral imagery with varying spatial and temporal resolutions, enabling continuous monitoring of deforestation and forest degradation across the Amazon.

In [ ]:
Deter['SATELLITE'].unique()

The dataset also includes information on the sensors used for image acquisition. The main sensors are WFI (Wide Field Imager), AWFI (Advanced Wide Field Imager), and AWIFS (Advanced Wide Field Sensor). These sensors are designed for wide-area coverage, making them suitable for large-scale monitoring of forest changes such as deforestation, degradation, and burn scars. They capture multispectral imagery with moderate spatial resolution, enabling frequent observations and timely detection of environmental changes in the Amazon region.

In [ ]:
Deter['SENSOR'].unique()

The dataset also provides the distribution of deforestation and degradation alerts by municipality. São Félix do Xingu records the highest number of alerts (24,305), followed by Altamira (18,068), Porto Velho (14,594), Lábrea (11,103), and Apuí (9,904). In contrast, several municipalities such as Anajatuba, Buriti do Tocantins, and Alto Taquari report only a single alert. In total, alerts span 586 municipalities, indicating that deforestation and related activities are widespread but highly concentrated in a few key regions within the Amazon.

In [ ]:
Deter['MUNICIPALI'].value_counts()

These lines process the VIEW_DATE column to extract temporal information for analysis. First, pd.to_datetime() converts the VIEW_DATE field into a proper datetime format, enabling time-based operations. Then, three new columns are created: year, extracted using .dt.year; month, using .dt.month; and year_month, using .dt.to_period('M') to represent each record as a specific year-month period (e.g., 2024-10)

In [ ]:
Deter.isnull().value_counts()

In [ ]:
# Parse dates and create time-related fields
Deter['VIEW_DATE']=pd.to_datetime(Deter['VIEW_DATE'])
Deter['year']=Deter['VIEW_DATE'].dt.year
Deter['month']=Deter['VIEW_DATE'].dt.month
Deter['year_month']=Deter['VIEW_DATE'].dt.to_period('M')

In [ ]:
Deter.head(5)

**b) Temperature Data**

To go beyond simply mapping forest loss, and actually measure its climatic consequences. By linking deforestation to temperature, you show whether and how land-use change is driving local warming, both immediately and with lagged effects.

Extracting Temperature Data from Google Earth Engine

This script uses Google Earth Engine to download daily temperature data for Porto Velho, Brazil, from 2016 to mid-2025. It first finds the boundary of Porto Velho, then collects ERA5 climate data (minimum, average, and maximum temperatures). Since ERA5 data is in Kelvin, the code converts it to Celsius. It samples the temperature values across the region, adds location (longitude/latitude), and prepares a preview table of a few rows. Finally, it exports the full dataset as a CSV file to Google Drive for further use.

In [ ]:
import ee, pandas as pd
# ee.Authenticate()
# ee.Initialize(project='ee-asimfulzele16')

# START_DATE  = '2016-01-01'
# END_DATE    = '2025-05-31'
# MAX_IMAGES  = None           # <-- include ALL days in range
# PREVIEW_ROWS = 15

# def get_porto_velho_geom():
#     gaul2 = ee.FeatureCollection('FAO/GAUL/2015/level2')
#     cand = (gaul2
#             .filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
#             .filter(ee.Filter.Or(
#                 ee.Filter.eq('ADM1_NAME', 'Rondônia'),
#                 ee.Filter.eq('ADM1_NAME', 'Rondonia')))
#             .filter(ee.Filter.eq('ADM2_NAME', 'Porto Velho')))
#     if cand.size().gt(0).getInfo():
#         return cand.first().geometry()
#     gaul2s = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2')
#     cand2 = (gaul2s
#              .filter(ee.Filter.eq('ADM0_NAME', 'Brazil'))
#              .filter(ee.Filter.Or(
#                  ee.Filter.eq('ADM1_NAME', 'Rondônia'),
#                  ee.Filter.eq('ADM1_NAME', 'Rondonia')))
#              .filter(ee.Filter.eq('ADM2_NAME', 'Porto Velho')))
#     if cand2.size().gt(0).getInfo():
#         return cand2.first().geometry()
#     return ee.Geometry.Point([-63.9039, -8.7608]).buffer(50000).bounds()

# porto_velho = get_porto_velho_geom()

# era = (ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR')
#        .filterDate(START_DATE, END_DATE)
#        .select(['temperature_2m_min', 'temperature_2m', 'temperature_2m_max']))

# def k_to_c(img):
#     c = ee.Image(img).subtract(273.15)
#     new_names = ee.Image(img).bandNames().map(lambda b: ee.String(b).cat('_c'))
#     return c.rename(new_names).copyProperties(img, ee.Image(img).propertyNames())

# era_c = era.map(k_to_c)
# proj = ee.Image(era_c.first()).projection()

# def sample_image(el):
#     img = ee.Image(el)
#     date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
#     lonlat = ee.Image.pixelLonLat().reproject(proj)
#     sampled = (img.addBands(lonlat)
#                  .sample(region=porto_velho,
#                          projection=proj,
#                          scale=proj.nominalScale(),
#                          geometries=True,
#                          tileScale=2))
#     return sampled.map(lambda f: f.set('date', date_str))

# img_list = era_c.toList(era_c.size())
# if MAX_IMAGES is not None:
#     img_list = img_list.slice(0, MAX_IMAGES)

# fc = ee.FeatureCollection(img_list.map(sample_image)).flatten()

# # Keep numeric lon/lat columns instead of geometry objects (smaller export)
# fc = fc.map(lambda f: f.set({
#     'lon': f.geometry().coordinates().get(0),
#     'lat': f.geometry().coordinates().get(1)
# }).setGeometry(None))

# # Preview a few rows (local)
# preview = fc.limit(PREVIEW_ROWS).getInfo()['features']
# rows = []
# for feat in preview:
#     p = feat['properties']
#     rows.append({
#         'date': p['date'],
#         'lon': p['lon'],
#         'lat': p['lat'],
#         't2m_min_c': p['temperature_2m_min_c'],
#         't2m_c': p['temperature_2m_c'],
#         't2m_max_c': p['temperature_2m_max_c'],
#     })
# print(pd.DataFrame(rows))

# # Export EVERYTHING
# selectors = ['date','lon','lat','temperature_2m_min_c','temperature_2m_c','temperature_2m_max_c']
# task = ee.batch.Export.table.toDrive(
#     collection = fc,
#     description = f"ERA5Land_PortoVelho_{START_DATE}_to_{END_DATE}",
#     fileFormat = 'CSV',
#     selectors = selectors
# )
# task.start()
# print("[INFO] Drive export started:", task.id)


Loading the extracted Temperature data

In [ ]:
ERA5=pd.read_csv('/content/drive/MyDrive/ERA5Land_PortoVelho_2016-01-01_to_2025-05-31.csv')

**date** → the day of observation (e.g., 2017-06-15).

**lon**→ longitude coordinate of the sample point.

**lat** → latitude coordinate of the sample point.

**temperature_2m_min_c **→ minimum daily air temperature (°C) measured 2 meters above ground.

**temperature_2m_c** → average daily air temperature (°C) at 2 meters above ground.

**temperature_2m_max_c** → maximum daily air temperature (°C) at 2 meters above ground.

Each row in the dataset represents one point in Porto Velho on a given date, with its location and the corresponding daily temperature values.

In [ ]:
ERA5

# **DATA PREPARATION**

**a) Deforestation Data**

This table shows the monthly aggregated area (in km²) of different forest disturbance classes detected by DETER, organized by year_month. Each row represents a month from August 2016 to June 2025, and columns indicate disturbance types such as CICATRIZ_DE_QUEIMADA (burn scars), CORTE_SELETIVO (selective logging), CS_DESORDENADO/CS_GEOMETRICO (types of selective cutting), DEGRADACAO (degradation), DESMATAMENTO_CR (clear-cut deforestation), DESMATAMENTO_VEG (vegetation removal), and MINERACAO (mining). For example, in August 2016, burn scars cover 9,285.76 km², and clear-cut deforestation accounts for 1,009.71 km², while other classes have smaller areas. Recent months like May 2025 show notable increases, particularly in clear-cutting (457.77 km²) and vegetation removal (489.48 km²). This data is crucial for identifying temporal trends, seasonal variations, and recent spikes in deforestation activities.

In [ ]:
monthly_agg =(Deter.groupby(['year_month','CLASSNAME'])['AREAMUNKM'].sum().reset_index().sort_values(by='year_month'))

In [ ]:
monthly_pivot=monthly_agg.pivot(index='year_month',columns='CLASSNAME',values='AREAMUNKM').fillna(0)

In [ ]:
monthly_pivot

This code converts the pivot table’s index to datetimes, sorts it chronologically, and builds an interactive stacked bar chart in Plotly where each disturbance CLASSNAME is a bar segment per month. It adds custom hovers showing month and area (km²), formats the x-axis as %Y-%m, and includes a range selector (1y, 2y, All) plus a range slider for zooming through 2016–2025. The layout uses the plotly_white theme, sets axis titles and legend, fixes the figure height, and finally renders the chart with fig.show().

In [ ]:
monthly_pivot.index = pd.to_datetime(monthly_pivot.index.astype(str))
monthly_pivot = monthly_pivot.sort_index()

fig = go.Figure()

# One stacked bar per CLASSNAME column
for classname in monthly_pivot.columns:
    fig.add_trace(go.Bar(
        x=monthly_pivot.index,
        y=monthly_pivot[classname],
        name=classname,
        hovertemplate="%{x|%Y-%m}: %{y:.2f} km²<extra>%{fullData.name}</extra>"
    ))

fig.update_layout(
    barmode='stack',
    title='Monthly Rates of Different Classes (DETER-B, 2016–2025)',
    xaxis=dict(
        title='Month',
        tickformat='%Y-%m',   # d3 time-format
        tickangle=45,
        rangeselector=dict(
            buttons=list([
                dict(count=12, step="month", stepmode="backward", label="1y"),
                dict(count=24, step="month", stepmode="backward", label="2y"),
                dict(step="all", label="All")
            ])
        ),
        rangeslider=dict(visible=True),
        type="date"
    ),
    yaxis_title='Area (km²)',
    template='plotly_white',
    legend_title='Class Name',
    height=600
)

fig.show()

This line filters the Deter GeoDataFrame to include only rows where the CLASSNAME is 'DESMATAMENTO_CR' (clear-cut deforestation). The resulting subset, stored in Deman, contains only deforestation alerts of this specific category, preserving all associated attributes and geometries. This step is essential for conducting focused analysis on clear-cut deforestation trends.

In [ ]:
Deman=Deter[Deter['CLASSNAME']=='DESMATAMENTO_CR']

In [ ]:
Deman.head()

In [ ]:
print(Deman.crs)

In [ ]:
Deman['MUNICIPALI'].value_counts()

This code filters the Deman GeoDataFrame to include only records where the municipality is Porto Velho, creating a new GeoDataFrame called Porto_velho. It then exports this filtered data as a shapefile named porto_velho_deforestation.shp to the specified Google Drive directory. This allows focused spatial analysis or visualization of clear-cut deforestation alerts specific to Porto Velho, while preserving all attribute data and geometries for GIS use

In [ ]:
Porto_velho=Deman[Deman['MUNICIPALI']=='Porto Velho']

In [ ]:
#Porto_velho.to_file("/content/drive/MyDrive/Asim/Dissertation/Porto_Velho/porto_velho_deforestation.shp")

In [ ]:
porto_velho=gpd.read_file('/content/drive/MyDrive/Asim/Dissertation/Porto_Velho/porto_velho_deforestation.shp')

In [ ]:
Porto_velho.head()

This code creates an interactive Folium map to visualize clear-cut deforestation alerts in Porto Velho by year. It starts by initializing the map centered on Porto Velho with CartoDB Positron basemap. A subset of relevant columns is selected, and VIEW_DATE is converted to string for display. Unique years are identified, and each year is assigned a distinct color using a predefined color palette. For each year, a FeatureGroup is created and populated with polygons representing deforestation alerts for that year. Each polygon is styled with its year-specific color and includes a tooltip showing year, area (km²), observation date, and municipality. Finally, all layers are added to the map with an interactive Layer Control, allowing users to toggle years on and off. The resulting map is saved as deforestation_by_year.html, providing an easy way to explore temporal patterns of deforestation in Porto Velho.

In [ ]:
m = folium.Map(location=[-9.573851, -64.663612], zoom_start=8, tiles='CartoDB positron')

columns_needed = ["MUNICIPALI", "UF", "VIEW_DATE", "SATELLITE", "SENSOR", "AREAMUNKM", "geometry","year"]
Porto_velho_min = Porto_velho[columns_needed].copy()
Porto_velho_min['VIEW_DATE']=Porto_velho_min['VIEW_DATE'].astype(str)
# Unique years in the dataset
years = sorted(Porto_velho_min['year'].unique())

# Assign a unique color to each year
color_palette = ('#8B0000','#00008B','#006400','#4B0082','#FF8C00','#800000','#4682B4','#228B22','#8B008B','#2F4F4F','#5F9EA0')

year_colors = {year: color_palette[i % len(color_palette)] for i, year in enumerate(years)}

# Create a dictionary of feature groups per year
year_layers = {}
for year in years:
    fg = folium.FeatureGroup(name=str(year))
    filtered = Porto_velho_min[Porto_velho_min['year'] == year]
    for _, row in filtered.iterrows():
        # Prepare popup content
        popup_html = f"""
        <strong>Year:</strong> {year}<br>
        <strong>Area:</strong> {row['AREAMUNKM']:.2f} km²<br>
        <strong>Date:</strong> {row['VIEW_DATE']}<br>
        <strong>Municipality:</strong> {row['MUNICIPALI']}
        """
        # Add GeoJson with popup and unique year color
        folium.GeoJson(
            row['geometry'],
            name=f'Deforestation {year}',
            style_function=lambda x, color=year_colors[year]: {
                'fillColor': color,
                'color': color,
                'weight': 1,
                'fillOpacity': 0.5
            },
            tooltip=folium.Tooltip(popup_html)
        ).add_to(fg)
    year_layers[year] = fg
    fg.add_to(m)

# Add layer control
folium.LayerControl(collapsed=False).add_to(m)

# Save or display
#m.save("deforestation_by_year.html")
m



This code aggregates deforestation alerts in Porto Velho by year and visualizes the results as a bar chart using Plotly. The groupby() function sums the AREAMUNKM (area in km²) for each year, creating the Porto_velho_yearly DataFrame. A bar trace is then added, with years on the x-axis and total deforested area on the y-axis, colored in green for clarity. The layout includes a descriptive title, labeled axes, and a clean white theme. The chart clearly shows annual variation in deforestation, with peaks around 2020–2022 and a sharp decline after 2023, indicating possible changes in land-use dynamics, enforcement measures, or reporting.

In [ ]:
Porto_velho_yearly=(Porto_velho.groupby(['year','MUNICIPALI'])['AREAMUNKM'].sum().reset_index())

fig = go.Figure()

fig.add_trace(go.Bar(
    x=Porto_velho_yearly['year'],
    y=Porto_velho_yearly['AREAMUNKM'],
    name='Yearly Deforestation',
    marker_color='forestgreen'
))

# Customize layout
fig.update_layout(
    title='Total Deforested Area in PORTO VELHO Municipality per Year',
    xaxis_title='Year',
    yaxis_title='Deforested Area (km²)',
    template='plotly_white',
    xaxis=dict(type='category'),  # Ensures years are treated as categories
    height=500
)

fig.show()

This code aggregates Porto Velho’s deforested area by year–month, fills in any missing months with zeros to keep a complete calendar, and adds readable month labels. It then builds a faceted area chart (one panel per year, wrapped 3 per row) showing monthly deforestation profiles, letting readers compare within-year seasonality and across-year changes at a glance. Peaks around Jul–Oct stand out in high-deforestation years, while recent years show much lower, flatter monthly totals.

In [ ]:
Porto_velho_monthly=(Porto_velho.groupby(['year','month'],as_index=False)['AREAMUNKM'].sum())

years = sorted(Porto_velho_monthly['year'].unique().tolist())
idx = pd.MultiIndex.from_product([years, range(1, 13)], names=['year', 'month'])

# Reindex to ensure every (year, month) exists
Porto_velho_monthly = (
    Porto_velho_monthly
        .set_index(['year', 'month'])
        .reindex(idx, fill_value=0)     # or .reindex(idx) then .fillna(...)
        .reset_index()
)

# Ensure month is integer
Porto_velho_monthly['month'] = Porto_velho_monthly['month'].astype(int)

# Nice month labels & ordering
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
month_map = dict(zip(range(1, 13), month_labels))

Porto_velho_monthly['month_name'] = pd.Categorical(
    Porto_velho_monthly['month'].map(month_map),
    categories=month_labels,
    ordered=True
)

# Grouped bar: months on x, color by year
fig = px.area(
    Porto_velho_monthly,
    x='month_name', y='AREAMUNKM',
    facet_col='year', facet_col_wrap=3,
    category_orders={'month_name': month_labels},
    title='Monthly Deforested Area Profiles — Porto Velho',
    labels={'month_name':'Month', 'AREAMUNKM':'Deforested Area (km²)'},
    height=600
)
fig.show()

**b) Temperature Data**

This code converts the date column to datetime, creates a month key, and then groups the ERA5 data by latitude, longitude, and month. It calculates monthly mean values for minimum, average, and maximum temperatures, counts the number of daily records per month, sorts the results, and prints the first few rows of the aggregated dataset.

In [ ]:
# Ensure datetime
ERA5["date"] = pd.to_datetime(ERA5["date"])

# Create month key (month start date)
ERA5["month"] = ERA5["date"].dt.to_period("M").dt.to_timestamp()

# Aggregate by lat, lon, month
monthly = (ERA5
    .groupby(["lat","lon","month"], as_index=False)
    .agg(
        t2m_min_c_mean=("temperature_2m_min_c","mean"),
        t2m_c_mean=("temperature_2m_c","mean"),
        t2m_max_c_mean=("temperature_2m_max_c","mean"),
        n_days=("date","nunique")   # daily obs count per month
    )
    .sort_values(["lat","lon","month"])
)
#monthly.to_csv('/content/drive/MyDrive/Asim/Monthly_Temp.csv')
print(monthly.head())

In [ ]:
Monthly_Temperature=pd.read_csv('/content/drive/MyDrive/Asim/Monthly_Temp.csv')

Extracted Temperature data is in the form of a CSV file, converting it to a GeoDataFrame

In [ ]:
Temperature = gpd.GeoDataFrame(
    Monthly_Temperature,
    geometry=gpd.points_from_xy(Monthly_Temperature['lon'], Monthly_Temperature['lat']),
    crs="EPSG:4326"
)
#Temperature.to_file('/content/drive/MyDrive/Asim/Temperature_PV.gpkg", driver="GPKG"')
# (optional) make sure date is datetime
#gdf['date'] = pd.to_datetime(gdf['date'])

This line of code loads a shapefile called Temperature_PV.shp into Python as a GeoDataFrame named Temperature_PV using GeoPandas.

In [ ]:
Temperature_PV=gpd.read_file('/content/drive/MyDrive/Asim/Temperature_PV.gpkg", driver="GPKG"/Temperature_PV.shp')

**month**: first day of the month

**t2m_min_c_ / t2m_c_mean / t2m_max_c**_: monthly means of min/avg/max daily temp (°C)

**n_days**: number of daily observations that month

**geometry**: the point location (lon, lat)

In [ ]:
Temperature_PV.head()

This line of code renames specific columns in the Temperature_PV DataFrame:

"t2m_max_c_" → "Maximum_Temp"

"t2m_c_mean" → "Mean_Temp"

"t2m_min_c_" → "Minimum_Temp"

In [ ]:
Temperature_PV = Temperature_PV.rename(columns={
    "t2m_max_c_": "Maximum_Temp",
    "t2m_c_mean": "Mean_Temp",
    "t2m_min_c_": "Minimum_Temp"
})

In [ ]:
Temperature_PV

**b) Deforested Data**

Weekly Deforested data

In [ ]:
PV_defor=gpd.read_file('/content/drive/MyDrive/Asim/Dissertation/Porto_Velho/Porto_velho_area/deforestation_weekly_polygons.gpkg')

This code takes a geospatial dataset of deforestation (PV_defor), makes a copy, and converts the week_start column into datetime format. It then creates a month column representing the first day of each month. Using GeoPandas’ dissolve, it aggregates data by month, spatially merging geometries (union) and summing deforested area (area_km2) across all weeks in the same month, while also counting how many weekly records contributed to each month (n_weeks). The result is a new GeoDataFrame (monthly) where each row represents total deforestation extent and combined geometry per month, with the original coordinate reference system (CRS) preserved.

In [ ]:
gdf = PV_defor.copy()
gdf["week_start"] = pd.to_datetime(gdf["week_start"])

# Month key (month start)
gdf["month"] = gdf["week_start"].dt.to_period("M").dt.to_timestamp()

# Dissolve by month:
# - geometry: union automatically (spatial dissolve)
# - area_km2: sum across the month's weeks
# - optionally count number of weekly parts that went into each month
monthly = gdf.dissolve(
    by="month",
    aggfunc={
        "area_km2": "sum",
        "week_start": "count"   # becomes a count of weekly records per month
    }
).rename(columns={"week_start": "n_weeks"}).reset_index()

# Keep CRS
monthly = gpd.GeoDataFrame(monthly, geometry="geometry", crs=gdf.crs)
#monthly.to_file('/content/drive/MyDrive/Asim/Deforestation_PV.gpkg", driver="GPKG"')
monthly

Loading the monthly deforestation data as a geopackage

In [ ]:
PV_defor=gpd.read_file('/content/drive/MyDrive/Asim/Deforestation_PV.gpkg", driver="GPKG"/Deforestation_PV.shp')

In [ ]:
PV_defor.head()

In [ ]:
PV_defor["month"] = pd.to_datetime(PV_defor["month"])

*Now, as both the datasets are on a same scale i.e monthly. So, now we combine both the datasets.*

**Temperature and Deforestation**

This code converts the month columns to datetime, then merges Temperature_PV and PV_defor on month, creating a combined dataset of monthly temperature and deforestation data with suffixes to distinguish overlapping column names.

In [ ]:
Temperature_PV["month"] = pd.to_datetime(Temperature_PV["month"])
PV_defor["month"]       = pd.to_datetime(PV_defor["month"])

combined = Temperature_PV.merge(PV_defor, on="month", suffixes=("_1","_2"))

In [ ]:
combined

**Monthly Deforestation vs Mean Temperature — Porto Velho (2017–2025)**

In [ ]:
from plotly.subplots import make_subplots

In [ ]:
combined["month"] = pd.to_datetime(combined["month"])

# Optional: restrict the study window (uncomment to enforce)
# start, end = "2017-01-01", "2025-05-31"
# msk = (combined["month"] >= start) & (combined["month"] <= end)
# combined = combined.loc[msk].copy()

# --- 1) Aggregate to regional monthly series ---------------------------------
monthly = (combined
           .groupby("month", as_index=False)
           .agg(Mean_Temp=("Mean_Temp","mean"),
                area_km2=("area_km2","sum")))

# Ensure regular monthly index (helps with plotting gaps)
monthly = monthly.set_index("month").sort_index()
# Optional: fill missing months (won’t fabricate values)
all_months = pd.date_range(monthly.index.min(), monthly.index.max(), freq="MS")
monthly = monthly.reindex(all_months)

# --- 2) Rolling means (optional; nice for clarity) ----------------------------
monthly["Mean_Temp_roll3"] = monthly["Mean_Temp"].rolling(3, min_periods=1).mean()
monthly["area_km2_roll3"] = monthly["area_km2"].rolling(3, min_periods=1).mean()

# --- 3) Build dual-axis figure ------------------------------------------------
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Temperature (left axis)
fig.add_trace(
    go.Scatter(
        x=monthly.index,
        y=monthly["Mean_Temp"],
        name="Mean Temp (°C)",
        mode="lines",
        line=dict(width=1.5)
    ),
    secondary_y=False
)
fig.add_trace(
    go.Scatter(
        x=monthly.index,
        y=monthly["Mean_Temp_roll3"],
        name="Mean Temp (3-mo MA)",
        mode="lines",
        line=dict(width=2, dash="dash")
    ),
    secondary_y=False
)

# Deforestation (right axis) – bars + smoothed line
fig.add_trace(
    go.Bar(
        x=monthly.index,
        y=monthly["area_km2"],
        name="Deforestation (km²)",
        opacity=0.4
    ),
    secondary_y=True
)
fig.add_trace(
    go.Scatter(
        x=monthly.index,
        y=monthly["area_km2_roll3"],
        name="Deforestation (3-mo MA)",
        mode="lines",
        line=dict(width=2)
    ),
    secondary_y=True
)

# --- 4) Layout & labels -------------------------------------------------------
fig.update_layout(
    title="Monthly Deforestation vs Mean Temperature — Porto Velho (2017–2025)",
    xaxis_title="Month",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    template="plotly_white",
    bargap=0.15,
    hovermode="x unified",
    margin=dict(l=60, r=60, t=70, b=50)
)

fig.update_yaxes(title_text="Mean Temperature (°C)", secondary_y=False)
fig.update_yaxes(title_text="Deforestation (km² / month)", secondary_y=True)

# --- 5) Show & save -----------------------------------------------------------
fig.show()

**Deforestation vs Temperature (Monthly, 2017–2025) with OLS Trendlines**

In [ ]:
combined_plot = combined.copy()
combined_plot["month"] = pd.to_datetime(combined_plot["month"])

# (Optional) aggregate to regional monthly: mean temps, sum area
monthly_scatter = (combined_plot
                   .groupby("month", as_index=False)
                   .agg(area_km2=("area_km2","sum"),
                        Mean_Temp=("Mean_Temp","mean"),
                        Maximum_Temp=("Maximum_Temp","mean"),
                        Minimum_Temp=("Minimum_Temp","mean"))
                  )

# Melt to long form for faceting (Mean/Max/Min)
melted = monthly_scatter.melt(
    id_vars=["month","area_km2"],
    value_vars=["Mean_Temp","Maximum_Temp","Minimum_Temp"],
    var_name="TempType", value_name="Temperature"
)

# Nice labels
label_map = {"Mean_Temp":"Mean Temp (°C)",
             "Maximum_Temp":"Maximum Temp (°C)",
             "Minimum_Temp":"Minimum Temp (°C)"}
melted["TempTypeLabel"] = melted["TempType"].map(label_map)

# Scatter with trendline (requires statsmodels installed)
fig2 = px.scatter(
    melted, x="area_km2", y="Temperature",
    color="TempTypeLabel", facet_col="TempTypeLabel", facet_col_wrap=1,
    trendline="ols", opacity=0.7,
    labels={"area_km2":"Deforestation (km² / month)", "Temperature":"Temperature (°C)"},
    title="Deforestation vs Temperature (Monthly, 2017–2025) with OLS Trendlines"
)

# Tidy layout
fig2.update_layout(
    template="plotly_white",
    showlegend=False,
    margin=dict(l=60, r=60, t=70, b=50),
    height=1000
)

# Show & save
fig2.show()
fig2.write_html("fig2_scatter_deforestation_vs_temps.html")
try:
    fig2.write_image("fig2_scatter_deforestation_vs_temps.png", scale=2, width=900, height=1000)
except Exception as e:
    print("[Note] PNG export needs kaleido: pip install -U kaleido. Error:", e)

This code turns the merged dataset into a GeoDataFrame using the geometry_2 column. It then reprojects the geometries into an equal-area projection (EPSG:6933) to avoid distortion, calculates the area of each geometry in square kilometers, and stores it in a new column area_km2_corrected. Finally, it prints a preview of the month, the original area_km2, and the corrected area values for comparison.

In [ ]:
gdf = gpd.GeoDataFrame(combined, geometry="geometry_2", crs="EPSG:4326")

# Reproject to equal-area CRS
gdf_equal = gdf.to_crs(epsg=6933)   # Cylindrical Equal Area projection

# Compute area in km²
gdf["area_km2_corrected"] = gdf_equal.geometry.area / 1e6

print(gdf[["month", "area_km2", "area_km2_corrected"]].head())

In [ ]:
gdf

This code sorts the GeoDataFrame by location (lat, lon) and time (month), then creates lagged versions of deforestation area (area_km2_corrected) to capture past values. For each spatial unit, it shifts the area values by 12, 24, and 36 months (1, 2, and 3 years) and stores them in new columns (area_km2_lag12, area_km2_lag24, area_km2_lag36). Finally, it prints the first 40 rows showing the current and lagged area values for comparison.

In [ ]:
gdf["month"] = pd.to_datetime(gdf["month"])
gdf = gdf.sort_values(["lat","lon","month"])

# Group by spatial unit (lat/lon or cell_id if you have it)
for lag in [12, 24, 36]:
    gdf[f"area_km2_lag{lag}"] = (
        gdf.groupby(["lat","lon"])["area_km2_corrected"]
           .shift(lag)
    )

print(gdf[["lat","lon","month","area_km2_corrected",
           "area_km2_lag12","area_km2_lag24","area_km2_lag36"]].head(40))

# **MODELLING**



# **1.   Temporal Analysis**




**Regression with yearly lags (Mean Temperature)**

This code builds and runs a linear regression model to test how past deforestation (lagged deforested area after 1, 2, and 3 years) relates to mean temperature. It sets the lagged area columns as predictors (X), the mean temperature as the dependent variable (y), adds a constant for the intercept, and fits an Ordinary Least Squares (OLS) regression using statsmodels. The model summary is then printed, showing coefficients, statistical significance, and goodness-of-fit metrics.

In [ ]:
import statsmodels.api as sm

X = gdf[["area_km2_lag12","area_km2_lag24","area_km2_lag36"]]
y = gdf["Mean_Temp"]

X = sm.add_constant(X)
model_year_lags = sm.OLS(y, X, missing="drop").fit()
print(model_year_lags.summary())

**Regression with yearly lags (Maximum Temperature)**

In [ ]:
# Regression with 1-3 year lags for Max Temp
y_max = gdf["Maximum_Temp"]
X = gdf[["area_km2_lag12","area_km2_lag24","area_km2_lag36"]]
X = sm.add_constant(X)

model_max = sm.OLS(y_max, X, missing="drop").fit()
print(model_max.summary())

**Regression with yearly lags (Minimum Temperature)**

In [ ]:
# Regression with 1-3 year lags for Minimum Temp
y_max = gdf["Minimum_Temp"]
X = gdf[["area_km2_lag12","area_km2_lag24","area_km2_lag36"]]
X = sm.add_constant(X)

model_max = sm.OLS(y_max, X, missing="drop").fit()
print(model_max.summary())

**AutoRegressive Distributed Lag model (ARDL)**

This code prepares a panel (cell-by-month) dataset for time-series regression. It creates a stable cell ID per (lat, lon), sorts by time, then adds: (a) deforestation lags at 12/24/36 months per cell, (b) autoregressive temperature terms (1-month and 12-month lags) for mean/max/min temperature, and (c) month fixed-effects via dummy variables to capture seasonality.

In [ ]:
gdf = gdf.copy()

# Ensure datetime & stable cell id
gdf["month"] = pd.to_datetime(gdf["month"])
gdf["cell_id"] = gdf.groupby(["lat","lon"], sort=False).ngroup()

# Sort within each cell by time
gdf = gdf.sort_values(["cell_id","month"])

# --- Create deforestation lags (years = 1,2,3 => 12/24/36 months) per cell ---
for k in [12, 24, 36]:
    gdf[f"defor_lag{k}"] = gdf.groupby("cell_id")["area_km2_corrected"].shift(k)

# --- Create temperature autoregressive terms (ARD part) per cell ---
# (use 1-month and 12-month lags; add more if you like)
for tgt in ["Mean_Temp","Maximum_Temp","Minimum_Temp"]:
    gdf[f"{tgt}_lag1"]  = gdf.groupby("cell_id")[tgt].shift(1)
    gdf[f"{tgt}_lag12"] = gdf.groupby("cell_id")[tgt].shift(12)

# Month fixed effects (seasonality)
gdf["month_num"] = gdf["month"].dt.month.astype("int8")
month_dummies = pd.get_dummies(gdf["month_num"], prefix="m", drop_first=True)
gdf = pd.concat([gdf, month_dummies], axis=1)

This function fits an ARDL-style OLS regression for a chosen temperature target (target_col). It builds predictors from deforestation lags (12/24/36 months), the target’s own lags (1 and 12 months), and month fixed effects (dummy vars m_*). It subsets to needed columns, coerces them to numeric, drops rows with any missing values, adds an intercept, and runs OLS with cluster-robust standard errors by cell_id to account for within-cell correlation. It returns the fitted statsmodels results object.

In [ ]:
def run_ardl(target_col: str):
    # predictors: deforestation lags + AR terms + month FE
    X_cols = [
        "defor_lag12", "defor_lag24", "defor_lag36",
        f"{target_col}_lag1", f"{target_col}_lag12",
    ] + [c for c in gdf.columns if c.startswith("m_")]

    # Keep only what we need
    dfm = gdf[["cell_id", target_col] + X_cols].copy()

    # Coerce everything (except cell_id) to numeric
    for c in [target_col] + X_cols:
        dfm[c] = pd.to_numeric(dfm[c], errors="coerce")

    # Drop rows with NaNs in target or predictors
    dfm = dfm.replace([np.inf, -np.inf], np.nan).dropna(subset=[target_col] + X_cols)

    # Build matrices
    X = sm.add_constant(dfm[X_cols], has_constant="add")
    y = dfm[target_col].to_numpy(dtype=float)

    # Fit with cluster-robust SEs by cell
    model = sm.OLS(y, X)
    res = model.fit(cov_type="cluster", cov_kwds={"groups": dfm["cell_id"]})
    return res

In [ ]:
month_fe = [c for c in gdf.columns if c.startswith("m_")]
for c in month_fe:
    gdf[c] = gdf[c].astype(np.int8)

This code runs the ARDL regression function three times—once each for Mean_Temp, Maximum_Temp, and Minimum_Temp. It stores the fitted results in res_mean, res_max, and res_min. Then, it prints the regression summaries, showing estimated coefficients, statistical significance, and model fit statistics for each temperature measure. This allows comparison of how past deforestation and autoregressive terms influence different temperature outcomes.

In [ ]:
res_mean = run_ardl("Mean_Temp")
res_max  = run_ardl("Maximum_Temp")
res_min  = run_ardl("Minimum_Temp")

print("=== ARDL (Mean_Temp) ===");   print(res_mean.summary())
print("\n=== ARDL (Maximum_Temp) ==="); print(res_max.summary())
print("\n=== ARDL (Minimum_Temp) ==="); print(res_min.summary())

***OLS vs ARDL Results Comparison Table***

This code extracts the coefficients, errors, and p-values for deforestation lags (12, 24, 36 months) from all three ARDL models, combines them into one table, adds significance stars, and prints a clear summary of lagged deforestation effects on temperature.

In [ ]:
def extract_coefs(res, model_name, lags=["defor_lag12","defor_lag24","defor_lag36"]):
    rows = []
    for lag in lags:
        if lag in res.params.index:
            coef = res.params[lag]
            se   = res.bse[lag]
            pval = res.pvalues[lag]
            rows.append({
                "Model": model_name,
                "Variable": lag,
                "Coef": coef,
                "SE": se,
                "pval": pval
            })
    return rows

# Collect all
all_coefs = []
all_coefs += extract_coefs(res_mean, "Mean_Temp")
all_coefs += extract_coefs(res_max, "Maximum_Temp")
all_coefs += extract_coefs(res_min, "Minimum_Temp")

coef_df = pd.DataFrame(all_coefs)

# Add significance stars
def stars(p):
    if p < 0.001: return "***"
    elif p < 0.01: return "**"
    elif p < 0.05: return "*"
    else: return ""
coef_df["Signif"] = coef_df["pval"].apply(stars)

print(coef_df)


***Coeficient with Error bars***

This code renames the lag variables into readable labels (1, 2, 3 years), then uses Plotly Express to create a grouped bar chart of the regression coefficients for each lag across the three temperature models. It adds error bars (±SE), displays significance stars above bars, and sets titles/labels so the plot clearly shows how past deforestation influences temperatures over time.

In [ ]:
import plotly.express as px

# Map lag names to years
lag_labels = {"defor_lag12": "Lag 12m (1yr)",
              "defor_lag24": "Lag 24m (2yr)",
              "defor_lag36": "Lag 36m (3yr)"}
coef_df["Lag"] = coef_df["Variable"].map(lag_labels)

fig = px.bar(
    coef_df,
    x="Lag", y="Coef",
    color="Model", barmode="group",
    error_y="SE",
    text="Signif",
    title="Deforestation Lag Effects on Temperature (ARDL)",
    labels={"Coef":"Coefficient (°C per km² deforestation)"}
)

fig.update_traces(textposition="outside")
fig.show()

***Time-Series Fit vs Observed***

This code prepares the dataset for Mean_Temp, keeping only relevant predictors (deforestation lags, autoregressive lags, and month dummies), drops rows with missing values, and builds the design matrix with an intercept. It then uses the fitted ARDL model (res_mean) to generate predicted temperature values and stores them in a new column Fitted for comparison with the observed Mean_Temp.

In [ ]:
# Example for Mean_Temp
dfm_mean = gdf[["cell_id","month","Mean_Temp","defor_lag12","defor_lag24","defor_lag36",
                "Mean_Temp_lag1","Mean_Temp_lag12"] + [c for c in gdf.columns if c.startswith("m_")]].copy()

# Drop missing rows
dfm_mean = dfm_mean.dropna()

# Build design matrix (same as in model)
X = dfm_mean[["defor_lag12","defor_lag24","defor_lag36","Mean_Temp_lag1","Mean_Temp_lag12"]
             + [c for c in gdf.columns if c.startswith("m_")]].to_numpy(dtype=float)
X = np.column_stack([np.ones(len(dfm_mean)), X])

# Predictions from ARDL
dfm_mean["Fitted"] = res_mean.predict(X)

In [ ]:
import plotly.graph_objects as go

# Pick one cell to visualize (otherwise too many lines overlap)
cell_id_example = dfm_mean["cell_id"].iloc[0]  # first cell
d = dfm_mean[dfm_mean["cell_id"] == cell_id_example]

fig = go.Figure()

# Observed temps
fig.add_trace(go.Scatter(
    x=d["month"], y=d["Mean_Temp"],
    mode="lines", name="Observed", line=dict(color="black")
))

# Fitted temps
fig.add_trace(go.Scatter(
    x=d["month"], y=d["Fitted"],
    mode="lines", name="ARDL Fitted", line=dict(color="red", dash="dash")
))

fig.update_layout(
    title=f"Observed vs ARDL Fitted Temperatures — Cell {cell_id_example}",
    xaxis_title="Time",
    yaxis_title="Mean Temperature (°C)",
    template="plotly_white"
)

fig.show()


This code compares observed vs. ARDL-predicted mean temperatures at the regional level. It averages both observed and fitted values by month, then plots them as time series: a solid black line for observed data and a dashed red line for model predictions. The chart clearly shows how well the ARDL model tracks temperature trends over time.

In [ ]:
regional = dfm_mean.groupby("month")[["Mean_Temp","Fitted"]].mean().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=regional["month"], y=regional["Mean_Temp"],
                         mode="lines", name="Observed", line=dict(color="black")))
fig.add_trace(go.Scatter(x=regional["month"], y=regional["Fitted"],
                         mode="lines", name="ARDL Fitted", line=dict(color="red", dash="dash")))

fig.update_layout(
    title="Observed vs ARDL Fitted Temperatures — Regional Average",
    xaxis_title="Time", yaxis_title="Mean Temperature (°C)",
    template="plotly_white"
)
fig.show()

***regional-average multi-panel figure***

Builds a helper to compute regional monthly Observed vs Fitted series (using the same ARDL design matrix) for any temperature target, generates those series for mean/max/min temps, and plots a 3-row subplot comparing observed lines to ARDL-fitted lines over time with shared x-axis and tidy layout.

In [ ]:
from plotly.subplots import make_subplots

# --- helper: build regional-average observed & fitted for a target ---
def regional_obs_fitted(gdf, res, target_col):
    # columns used in the ARDL model
    month_fe = [c for c in gdf.columns if c.startswith("m_")]
    X_cols = ["defor_lag12","defor_lag24","defor_lag36",
              f"{target_col}_lag1", f"{target_col}_lag12"] + month_fe

    dfm = gdf[["month", "cell_id", target_col] + X_cols].copy().dropna()

    # design matrix (must match model training order)
    X = dfm[["defor_lag12","defor_lag24","defor_lag36",
             f"{target_col}_lag1", f"{target_col}_lag12"] + month_fe].to_numpy(dtype=float)
    X = np.column_stack([np.ones(len(dfm)), X])  # add constant

    # fitted values
    dfm["Fitted"] = res.predict(X)

    # regional average by month
    reg = (dfm.groupby("month")[[target_col, "Fitted"]]
                .mean()
                .reset_index()
                .rename(columns={target_col: "Observed"}))
    return reg

# Build regional series for all three targets
reg_mean = regional_obs_fitted(gdf, res_mean, "Mean_Temp")
reg_max  = regional_obs_fitted(gdf, res_max,  "Maximum_Temp")
reg_min  = regional_obs_fitted(gdf, res_min,  "Minimum_Temp")

# --- multi-panel plot (3 rows, shared x) ---
fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True,
    vertical_spacing=0.06,
    subplot_titles=("Mean Temperature — Regional Average",
                    "Maximum Temperature — Regional Average",
                    "Minimum Temperature — Regional Average")
)

# panel 1: Mean
fig.add_trace(go.Scatter(x=reg_mean["month"], y=reg_mean["Observed"],
                         name="Observed (Mean)", mode="lines"),
              row=1, col=1)
fig.add_trace(go.Scatter(x=reg_mean["month"], y=reg_mean["Fitted"],
                         name="ARDL Fitted (Mean)", mode="lines", line=dict(dash="dash")),
              row=1, col=1)

# panel 2: Max
fig.add_trace(go.Scatter(x=reg_max["month"], y=reg_max["Observed"],
                         name="Observed (Max)", mode="lines"),
              row=2, col=1)
fig.add_trace(go.Scatter(x=reg_max["month"], y=reg_max["Fitted"],
                         name="ARDL Fitted (Max)", mode="lines", line=dict(dash="dash")),
              row=2, col=1)

# panel 3: Min
fig.add_trace(go.Scatter(x=reg_min["month"], y=reg_min["Observed"],
                         name="Observed (Min)", mode="lines"),
              row=3, col=1)
fig.add_trace(go.Scatter(x=reg_min["month"], y=reg_min["Fitted"],
                         name="ARDL Fitted (Min)", mode="lines", line=dict(dash="dash")),
              row=3, col=1)

fig.update_layout(
    height=900,
    title="Observed vs ARDL Fitted — Regional Monthly Averages",
    template="plotly_white",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0)
)

fig.update_xaxes(title_text="Month", row=3, col=1)
fig.update_yaxes(title_text="°C", row=1, col=1)
fig.update_yaxes(title_text="°C", row=2, col=1)
fig.update_yaxes(title_text="°C", row=3, col=1)

fig.show()

This code evaluates ARDL model performance by comparing observed vs fitted regional averages. It computes RMSE, MAE, and R² for each target (mean, maximum, minimum temperature), stores them in a summary table (metrics_df), and prints the results — giving a clear measure of predictive accuracy.

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def metrics(df, y_col="Observed", yhat_col="Fitted"):
    rmse = np.sqrt(mean_squared_error(df[y_col], df[yhat_col]))
    mae  = mean_absolute_error(df[y_col], df[yhat_col])
    r2   = r2_score(df[y_col], df[yhat_col])
    return rmse, mae, r2

summary = []
for name, reg in [("Mean_Temp", reg_mean), ("Maximum_Temp", reg_max), ("Minimum_Temp", reg_min)]:
    rmse, mae, r2 = metrics(reg)
    summary.append({"Target": name, "RMSE_°C": rmse, "MAE_°C": mae, "R2": r2})

metrics_df = pd.DataFrame(summary)
print(metrics_df)

**Model performance (Adjusted R²) — OLS vs ARDL-style**

In [ ]:
aos = combined.copy()
aos["month"] = pd.to_datetime(aos["month"])
aos = aos.sort_values("month").set_index("month")

# --- Create lags for ARDL-like model ---
for L in (12, 24, 36):
    aos[f"defor_lag{L}"] = aos["area_km2"].shift(L)

for tcol in ("Mean_Temp","Maximum_Temp","Minimum_Temp"):
    aos[f"{tcol}_lag1"] = aos[tcol].shift(1)
    aos[f"{tcol}_lag12"] = aos[tcol].shift(12)

# Drop rows with NA (introduced by lags)
aos_reg = aos.dropna().copy()

def fit_ols(ycol):
    X = sm.add_constant(aos_reg[["area_km2"]])
    y = aos_reg[ycol]
    return sm.OLS(y, X).fit().rsquared_adj

def fit_ardl_like(ycol):
    X = aos_reg[[f"{ycol}_lag1", f"{ycol}_lag12", "defor_lag12", "defor_lag24", "defor_lag36"]]
    X = sm.add_constant(X)
    y = aos_reg[ycol]
    return sm.OLS(y, X).fit().rsquared_adj

metrics = []
for ycol, label in [("Mean_Temp","Mean Temp"),
                    ("Maximum_Temp","Max Temp"),
                    ("Minimum_Temp","Min Temp")]:
    r2_ols = fit_ols(ycol)
    r2_ardl = fit_ardl_like(ycol)
    metrics.append({"Temp": label, "Model": "OLS", "AdjR2": r2_ols})
    metrics.append({"Temp": label, "Model": "ARDL-style", "AdjR2": r2_ardl})

perf = pd.DataFrame(metrics)

# --- Plot grouped bars ---
fig4 = go.Figure()
for model_name in ["OLS", "ARDL-style"]:
    fig4.add_bar(
        name=model_name,
        x=perf.loc[perf["Model"]==model_name, "Temp"],
        y=perf.loc[perf["Model"]==model_name, "AdjR2"]
    )

fig4.update_layout(
    barmode="group",
    template="plotly_white",
    title="Model Performance: Adjusted R² (OLS vs ARDL-style)",
    xaxis_title="Temperature Metric",
    yaxis_title="Adjusted R²",
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0),
    margin=dict(l=60, r=60, t=70, b=50)
)

fig4.show()
fig4.write_html("fig4_model_performance_ols_vs_ardl.html")
try:
    fig4.write_image("fig4_model_performance_ols_vs_ardl.png", scale=2, width=900, height=600)
except Exception as e:
    print("[Note] PNG export needs kaleido: pip install -U kaleido. Error:", e)

print(perf)

# **2. Spatial Analysis**



**Local Moran’s I with k-nearest neighbors**

In [ ]:
!pip install esda

In [ ]:
import geopandas as gpd
import pandas as pd
import statsmodels.api as sm
import numpy as np

# -------------------------------------------------------------------
# 1. Run your regressions again to create coef_gdf
# -------------------------------------------------------------------
results = []
for cell, df_cell in gdf.groupby("cell_id"):
    df_cell = df_cell.dropna(subset=["Mean_Temp","defor_lag12","defor_lag24","defor_lag36"])
    if len(df_cell) < 24:  # require at least 2 years of data
        continue

    X = df_cell[["defor_lag12","defor_lag24","defor_lag36"]]
    X = sm.add_constant(X)
    y = df_cell["Mean_Temp"]

    model = sm.OLS(y, X).fit()

    results.append({
        "cell_id": cell,
        "coef_lag12": model.params.get("defor_lag12", np.nan),
        "coef_lag24": model.params.get("defor_lag24", np.nan),
        "coef_lag36": model.params.get("defor_lag36", np.nan),
        "r2": model.rsquared,
        "n_obs": len(df_cell)
    })

coef_df = pd.DataFrame(results)

# -------------------------------------------------------------------
# 2. Merge back with geometry
# -------------------------------------------------------------------
gdf_unique = gdf.drop_duplicates("cell_id")[["cell_id","geometry_2"]]
coef_gdf = gdf_unique.merge(coef_df, on="cell_id")

# Make it a GeoDataFrame
coef_gdf = gpd.GeoDataFrame(coef_gdf, geometry="geometry_2", crs=gdf.crs or "EPSG:4326")


Creates a GeoDataFrame of per-cell coefficients, builds k-nearest-neighbor spatial weights, computes global Moran’s I and local Moran’s I (LISA) for coef_lag36, classifies each cell into hotspot/coldspot/outlier/not-significant, then renders both a static matplotlib map and an interactive Folium map (with tooltips, legend, and controls). Finally saves the interactive map to lisa_hotspots_coef_lag36_knn.html.

In [ ]:
# -*- coding: utf-8 -*-
# Full workflow: Global & Local Moran's I + Static Plot + Folium Map with Basemaps

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from libpysal.weights import KNN
from esda.moran import Moran, Moran_Local

import folium
from folium import plugins
import branca

# -------------------------------------------------------------------
# 0) INPUT: coef_gdf (one row per cell_id)
#    Must have at least: ['cell_id','coef_lag36','geometry_2']
#    Assumes coef_gdf already exists in memory.
# -------------------------------------------------------------------

# Ensure active geometry and a known CRS
coef_gdf = gpd.GeoDataFrame(coef_gdf.copy(), geometry="geometry_2", crs=coef_gdf.crs or "EPSG:4326")
if coef_gdf.crs is None:
    coef_gdf = coef_gdf.set_crs(epsg=4326)

# Project to metric CRS for neighbor geometry (k-NN) calculations
coef_proj = coef_gdf if (coef_gdf.crs.to_epsg() and coef_gdf.crs.to_epsg() != 4326) else coef_gdf.to_crs(3857)

# Target variable
y_all = coef_proj["coef_lag36"].astype(float).to_numpy()
valid = np.isfinite(y_all)
coef_proj = coef_proj.loc[valid].copy()
y = y_all[valid]

# -------------------------------------------------------------------
# 1) Spatial weights: k-nearest neighbors (tune k if hotspots look too small)
# -------------------------------------------------------------------
k = 8  # try 10–12 for larger clusters
w = KNN.from_dataframe(coef_proj, k=k)
w.transform = "R"  # row-standardized

# -------------------------------------------------------------------
# 2) Global Moran's I
# -------------------------------------------------------------------
mi = Moran(y, w, permutations=999)
print("=== Global Moran's I (coef_lag36) ===")
print(f"I = {mi.I:.4f}, z = {mi.z_norm:.2f}, p (perm) = {mi.p_sim:.4f}")

# -------------------------------------------------------------------
# 3) Local Moran's I (LISA)
# -------------------------------------------------------------------
np.random.seed(42)
mli = Moran_Local(y, w, permutations=999)

alpha = 0.05
sig = mli.p_sim < alpha
quad = mli.q  # 1=HH, 2=LH, 3=LL, 4=HL

# Build class labels
labels = np.array(["Not significant"] * coef_proj.shape[0], dtype=object)
labels[(quad == 1) & sig] = "High-High (Hotspot)"
labels[(quad == 3) & sig] = "Low-Low (Coldspot)"
labels[(quad == 4) & sig] = "High-Low (Outlier)"
labels[(quad == 2) & sig] = "Low-High (Outlier)"

coef_proj["LISA_class"] = labels
coef_proj["LISA_I"]     = mli.Is
coef_proj["LISA_p"]     = mli.p_sim
coef_proj["LISA_z"]     = mli.z_sim

# -------------------------------------------------------------------
# 4) Static map (matplotlib)
# -------------------------------------------------------------------
order = ["High-High (Hotspot)", "Low-Low (Coldspot)", "High-Low (Outlier)", "Low-High (Outlier)", "Not significant"]
palette = {
    "High-High (Hotspot)": "#b2182b",
    "Low-Low (Coldspot)":  "#2166ac",
    "High-Low (Outlier)":  "#f4a582",
    "Low-High (Outlier)":  "#92c5de",
    "Not significant":     "#cccccc",
}
coef_proj["LISA_class"] = pd.Categorical(coef_proj["LISA_class"], categories=order, ordered=True)

fig, ax = plt.subplots(figsize=(9, 8))
coef_proj.plot(
    ax=ax,
    color=coef_proj["LISA_class"].map(palette),
    linewidth=0.2,
    edgecolor="#555555"
)
ax.set_title(f"Hotspot Map — Local Moran's I on coef_lag36 (k={k})", fontsize=14)
ax.set_axis_off()

handles = [mpatches.Patch(color=palette[c], label=c) for c in order]
ax.legend(handles=handles, title="LISA class", loc="lower left", frameon=True)
plt.tight_layout()
plt.show()

# -------------------------------------------------------------------
# 5) Interactive Folium map with multiple basemaps
# -------------------------------------------------------------------
# Prepare WGS84 and (optionally) simplified geometry for web rendering
coef_wgs = coef_proj.to_crs(4326).copy()
if coef_wgs.geometry.name != "geometry":
    coef_wgs = coef_wgs.rename_geometry("geometry")  # rename only if different

# (Optional) simplify polygons for web speed (~50 m in Web Mercator then back)
coeff_simple = coef_wgs.to_crs(3857).copy()
coeff_simple = coeff_simple.set_geometry(coeff_simple.geometry.simplify(50))  # adjust tolerance as needed
coef_wgs_simplified = coeff_simple.to_crs(4326)
if coef_wgs_simplified.geometry.name != "geometry":
    coef_wgs_simplified = coef_wgs_simplified.rename_geometry("geometry")

# Map center
minx, miny, maxx, maxy = coef_wgs_simplified.total_bounds
center = [(miny + maxy) / 2, (minx + maxx) / 2]

# Create map with no default tiles; add multiple basemaps
m = folium.Map(location=center, zoom_start=7, control_scale=True, tiles=None)

# Common basemaps
folium.TileLayer('OpenStreetMap', name='OpenStreetMap').add_to(m)
folium.TileLayer('CartoDB positron', name='CartoDB Positron').add_to(m)
folium.TileLayer('CartoDB dark_matter', name='CartoDB Dark Matter').add_to(m)

# Stamen Terrain (explicit URL + attribution)
folium.TileLayer(
    tiles="https://stamen-tiles.a.ssl.fastly.net/terrain/{z}/{x}/{y}.jpg",
    attr="Map tiles by Stamen Design, CC BY 3.0 — Map data © OpenStreetMap contributors",
    name="Stamen Terrain"
).add_to(m)

# High-res satellite (Esri World Imagery)
folium.TileLayer(
    tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}',
    attr='&copy; Esri &mdash; Sources: Esri, i-cubed, USDA, USGS, AEX, GeoEye, Getmapping, Aerogrid, IGN, IGP, UPR-EGP, and the GIS User Community',
    name='Esri World Imagery'
).add_to(m)

# Class colors (same as static)
palette = {
    "High-High (Hotspot)": "#b2182b",
    "Low-Low (Coldspot)":  "#2166ac",
    "High-Low (Outlier)":  "#f4a582",
    "Low-High (Outlier)":  "#92c5de",
    "Not significant":     "#cccccc",
}
order = list(palette.keys())

def class_color(c):
    return palette.get(c, "#cccccc")

def style_fn(feat):
    cls = feat["properties"].get("LISA_class", "Not significant")
    return {
        "fillColor": class_color(cls),
        "color": "#555555",
        "weight": 0.5,
        "fillOpacity": 0.65,  # adjust transparency if needed
    }

tooltip = folium.GeoJsonTooltip(
    fields=["cell_id", "coef_lag36", "LISA_I", "LISA_p", "LISA_class"],
    aliases=["Cell ID", "β lag36", "Local I", "p-value", "Class"],
    localize=True, sticky=False, labels=True
)

folium.GeoJson(
    data=coef_wgs_simplified[["cell_id", "coef_lag36", "LISA_I", "LISA_p", "LISA_class", "geometry"]].__geo_interface__,
    name="LISA Hotspots (coef_lag36)",
    style_function=style_fn,
    tooltip=tooltip,
    show=True
).add_to(m)

# Categorical legend
legend_html = f"""
<div style="position: fixed; bottom: 20px; left: 20px; width: 260px; z-index:9999;
            background-color: white; padding: 10px; border:1px solid #ccc; border-radius:6px;">
<b>Local Moran's I</b><br>
"""
for cls in order:
    legend_html += f'<i style="background:{palette[cls]};width:12px;height:12px;display:inline-block;margin-right:6px;"></i>{cls}<br>'
legend_html += "</div>"
m.get_root().html.add_child(folium.Element(legend_html))

# Controls & handy plugins
folium.LayerControl(collapsed=False).add_to(m)
try:
    plugins.Fullscreen().add_to(m)
    plugins.MeasureControl(position="topleft", primary_length_unit="kilometers").add_to(m)
    plugins.ScaleBar(position="bottomleft").add_to(m)
except Exception:
    pass

# Save
m.save("lisa_hotspots_coef_lag36_knn.html")
print("Saved -> lisa_hotspots_coef_lag36_knn.html")

# For notebooks, also display the map object:
m


In [ ]:
# ===================================================================================
# GENERATE REQUIREMENTS FILE
#
# Before you submit, you must run this code in a final cell in your Code_Notebook,
# NOT INSIDE THIS COVERSHEET_NOTEBOOK. This will create the requirements.json
# file which is required for marking.
#
# Make sure you have followed the instructions above to run your
# entire Code_Notebook from top to bottom BEFORE running this code.
# ===================================================================================

import json
from importlib import metadata
import sys
import datetime

print("Generating requirements.json file...")

try:
    # Get all installed packages and their version numbers
    installed_packages = {dist.metadata['Name']: dist.metadata['Version']
                          for dist in metadata.distributions()}

    # Sort the package list alphabetically
    sorted_packages = dict(sorted(installed_packages.items()))

    # Get the Python version
    python_version = f"{sys.version_info.major}.{sys.version_info.minor}.{sys.version_info.micro}"

    # Get the current timestamp in UTC
    generation_timestamp_utc = datetime.datetime.now(datetime.timezone.utc).isoformat() # <--- ADDED

    # Combine all data into a single dictionary
    environment_data = {
        "generation_timestamp_utc": generation_timestamp_utc, # <--- ADDED
        "python_version": python_version,
        "packages": sorted_packages
    }

    # Define the output file name
    file_name = "requirements.json"

    # Write the data to the requirements.json file in the current directory
    with open(file_name, 'w') as f:
        json.dump(environment_data, f, indent=4)

    print("-" * 50)
    print(f"SUCCESS: The '{file_name}' file has been created successfully.")
    print("Please check the file is present and as expected before you submit.")
    print("-" * 50)

except Exception as e:
    print("-" * 50)
    print(f"ERROR: An error occurred while generating the file.")
    print("Please contact a member of the teaching team for help, including any errors or warnings.")
    print(f"Error details: {e}")
    print("-" * 50)